In [1]:
import pyranges1 as pr
import pandas as pd
import subprocess
from pathlib import Path
import gzip
import pysam
from contextlib import nullcontext
from Bio.SeqUtils import seq3
import re

In [2]:
pd.set_option('display.max_columns', None)

## GFF preparation + BED preparation

In [ ]:
gtf_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.gtf.gz'

In [ ]:
command = f'''./prepare_data -i {gtf_filepath} -rm 1 --add_chr'''
subprocess.run(command, shell=True, check=True)

## MNV calculation from VCF

In [ ]:
input_vcf_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.vcf.gz'
fasta_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.fa'
bam_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.bam.genozip'
reference_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.ref.genozip'
bed_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/cds.bed'
cds_table_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/cds_table.parquet'
output_vcf_filepath = input_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.mnv.vcf.gz'

command = f'''./cis2mnv -i {input_vcf_filepath} \
        --fasta {fasta_filepath} \
        --bam {bam_genozip_filepath} \
        --genozip_ref {reference_genozip_filepath} \
        --bed {bed_filepath} \
        --merge_overlapped_regions 0 \
        --only_pass_variants 1 \
        --cds_table {cds_table_filepath} \
        --remove_intermediate_files 1 \
        --compress 1 \
        -o {output_vcf_filepath}'''
subprocess.run(command, shell=True, check=True)

## VCF annotation

### VEP

In [ ]:
gtf_single_transcripts_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.main_transcripts.gtf.gz'
vep_annotated_vcf_filepath = output_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.vep.vcf.gz'

command = f'''mamba run -n cis2mnv-vep.venv vep \
    --input_file {output_vcf_filepath} \
    --output_file {vep_annotated_vcf_filepath} \
    --gtf {gtf_single_transcripts_filepath} \
    --fasta {fasta_filepath} \
    --warning_file STDERR \
    --no_stats \
    --force_overwrite \
    --format vcf \
    --vcf \
    --compress_output bgzip \
    --symbol \
    --protein \
    --hgvs'''
subprocess.run(command, shell=True, check=True)

### SnpEff

In [ ]:
snpeff_annotated_vcf_filepath = output_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.snpeff.vcf.gz'

command = f'''mamba run -n cis2mnv-snpeff.venv snpeff ann \
    -canon \
    -noStats \
    -download hg38 {output_vcf_filepath} \
    | bgzip -c > {snpeff_annotated_vcf_filepath}'''
subprocess.run(command, shell=True, check=True)

## TESTS

### Transitions

#### Ensembl sourses

In [ ]:
ensembl_refseq_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/dbs/Homo_sapiens.GRCh38.116.refseq.tsv.gz'
ensembl_uniprot_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/dbs/Homo_sapiens.GRCh38.116.uniprot.tsv.gz'

In [ ]:
ensembl_refseq_df = pd.read_csv(ensembl_refseq_filepath, sep='\t')
ensembl_uniprot_df = pd.read_csv(ensembl_uniprot_filepath, sep='\t')

ensembl_refseq_df = (ensembl_refseq_df
                     .query('db_name == "RefSeq_mRNA" and info_type == "DIRECT" and source_identity == "100" and xref_identity == "100"')
                     [['transcript_stable_id', 'xref']]
                     .rename(columns={'xref': 'refseq_id'}))

ensembl_uniprot_df = (ensembl_uniprot_df
                     .query('info_type == "DIRECT" and source_identity == "100" and xref_identity == "100"')
                     [['transcript_stable_id', 'xref']]
                     .rename(columns={'xref': 'uniprot_id'}))

transitions_df = ensembl_refseq_df.merge(ensembl_uniprot_df, on='transcript_stable_id', how='outer').rename(columns={'transcript_stable_id': 'ensembl_id'})

In [ ]:
transitions_df.sort_values(by='ensembl_id')

In [ ]:
transitions_df.memory_usage(deep=True).sum() / 1024**2

### ClinVar

In [ ]:
def prepare_clinvar(input_filpath, output_filepath, assembly='GRCh38'):
    '''assemblies available: GRCh37 / GRCh38'''

    clinvar_df = pd.read_csv(input_filpath, sep='\t', usecols=['Name', 'ClinicalSignificance', 'Assembly'], )

    clinvar_df_filtered = (clinvar_df[clinvar_df['Assembly'] == assembly]
                       .replace('-', pd.NA)
                       .dropna(subset=['Name', 'ClinicalSignificance'], how='any'))

    clinvar_df_filtered[['HGVSc', 'HGVSp']] = clinvar_df_filtered['Name'].str.split(' ', n=1, expand=True).reindex(columns=[0, 1])
    clinvar_df_filtered[['transcript', 'HGVSc']] = clinvar_df_filtered['HGVSc'].str.split(':', n=1, expand=True).reindex(columns=[0, 1])
    clinvar_df_filtered[['transcript', 'gene_name']] = clinvar_df_filtered['transcript'].str.split('(', n=1, expand=True).reindex(columns=[0, 1])
    clinvar_df_filtered[['transcript', 'transcript_version']] = clinvar_df_filtered['transcript'].str.split('.', n=1, expand=True).reindex(columns=[0, 1])
    clinvar_df_filtered['HGVSp'] = clinvar_df_filtered['HGVSp'].str.strip('()')
    clinvar_df_filtered['gene_name'] = clinvar_df_filtered['gene_name'].str.strip(')')

    clinvar_df_short = (clinvar_df_filtered[['gene_name', 'transcript', 'transcript_version', 'HGVSc', 'HGVSp', 'ClinicalSignificance']]
                    .dropna(how='any', axis=0)
                    .query('HGVSp.str.startswith("p.") and ~HGVSp.str.contains("fs|Ter|=|[?]") and ~HGVSp.str.endswith("del")', engine='python'))
    
    clinvar_df_short = (clinvar_df_short.groupby(['transcript', 'HGVSp'], as_index=False)['ClinicalSignificance']
                             .agg(lambda values: '; '.join(dict.fromkeys(
                                item.strip()
                                for value in values.astype(str)
                                for item in value.split(';')
                                if item.strip()
                            ))))

    clinvar_df_short['protein_position'] = (clinvar_df_short['HGVSp']
                                            .apply(extract_protein_position)
                                            .astype('Int64'))

    transitions = {
    'Uncertain significance': 'U',
    'Conflicting classifications of pathogenicity': 'C',
    'Likely benign': 'LB',
    'Likely pathogenic': 'LP',
    'Benign': 'B',
    'Pathogenic': 'P',
    'not provided': 'NP',
    'other': 'O',
    'drug response': 'DR',
    'no classification for the single variant': 'NC',
    'risk factor': 'RF',
    'association': 'As',
    'Affects': 'Af',
    }

    def replace_clin_sig(string, replace_dict):
        for key, value in replace_dict.items():
            string = string.replace(key, value)
        return string

    clinvar_df_short['ClinicalSignificance'] = clinvar_df_short['ClinicalSignificance'].apply(replace_clin_sig, args=(transitions, ))

    clinvar_df_short = clinvar_df_short[['transcript', 'protein_position', 'HGVSp', 'ClinicalSignificance']]

    clinvar_df_short.to_csv(output_filepath, sep='\t', compression='gzip', index=False)

    

In [3]:
clinvar_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/dbs/variant_summary.txt.gz'
assembly = 'GRCh38' # GRCh37 / GRCh38

In [ ]:
clinvar_df = pd.read_csv(clinvar_filepath, sep='\t', usecols=['Name', 'ClinicalSignificance', 'Assembly'])
#clinvar_df = pd.read_csv(clinvar_filepath, sep='\t', usecols=['Name', 'ClinicalSignificance', 'Assembly'], dtype={'ClinicalSignificance': 'category', 'Assembly': 'category'})

In [ ]:
clinvar_df_filtered = (clinvar_df[clinvar_df['Assembly'] == assembly]
                       .replace('-', pd.NA)
                       .dropna(subset=['Name', 'ClinicalSignificance'], how='any'))

In [ ]:
clinvar_df_filtered[['HGVSc', 'HGVSp']] = clinvar_df_filtered['Name'].str.split(' ', n=1, expand=True).reindex(columns=[0, 1])
clinvar_df_filtered[['transcript', 'HGVSc']] = clinvar_df_filtered['HGVSc'].str.split(':', n=1, expand=True).reindex(columns=[0, 1])
clinvar_df_filtered[['transcript', 'gene_name']] = clinvar_df_filtered['transcript'].str.split('(', n=1, expand=True).reindex(columns=[0, 1])
clinvar_df_filtered[['transcript', 'transcript_version']] = clinvar_df_filtered['transcript'].str.split('.', n=1, expand=True).reindex(columns=[0, 1])

clinvar_df_filtered['HGVSp'] = clinvar_df_filtered['HGVSp'].str.strip('()')
clinvar_df_filtered['gene_name'] = clinvar_df_filtered['gene_name'].str.strip(')')

In [ ]:
clinvar_df_short = (clinvar_df_filtered[['gene_name', 'transcript', 'transcript_version', 'HGVSc', 'HGVSp', 'ClinicalSignificance']]
                    .dropna(how='any', axis=0)
                    .query('HGVSp.str.startswith("p.") and ~HGVSp.str.contains("fs|Ter|=|[?]") and ~HGVSp.str.endswith("del")', engine='python'))

In [ ]:
clinvar_df_short = (clinvar_df_short.groupby(['transcript', 'HGVSp'], as_index=False)['ClinicalSignificance']
                             .agg(lambda values: '; '.join(dict.fromkeys(
                                item.strip()
                                for value in values.astype(str)
                                for item in value.split(';')
                                if item.strip()
                            ))))

In [ ]:
def extract_protein_position(value):
    return int(re.search(r'\d+', value).group())

clinvar_df_short['protein_position'] = clinvar_df_short['HGVSp'].apply(extract_protein_position)

In [ ]:
transitions = {
    'Uncertain significance': 'U',
    'Conflicting classifications of pathogenicity': 'C',
    'Likely benign': 'LB',
    'Likely pathogenic': 'LP',
    'Benign': 'B',
    'Pathogenic': 'P',
    'not provided': 'NP',
    'other': 'O',
    'drug response': 'DR',
    'no classification for the single variant': 'NC',
    'risk factor': 'RF',
    'association': 'As',
    'Affects': 'Af',
}

def replace_clin_sig(string, replace_dict):
    for key, value in replace_dict.items():
        string = string.replace(key, value)
    return string

clinvar_df_short['ClinicalSignificance'] = clinvar_df_short['ClinicalSignificance'].apply(replace_clin_sig, args=(transitions, ))

In [ ]:
clinvar_df_short = clinvar_df_short[['transcript', 'protein_position', 'HGVSp', 'ClinicalSignificance']]

In [ ]:
clinvar_df_short

In [ ]:
clinvar_df.memory_usage(deep=True).sum() / 1024**2

In [ ]:
clinvar_df_short.memory_usage(deep=True).sum() / 1024**2

### AlphaMissene

In [ ]:
alphamissense_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/dbs/AlphaMissense_aa_substitutions.tsv.gz'

In [ ]:
def extract_protein_position(value):
    return int(re.search(r'\d+', value).group())

def to_hgvsp(value):
    return f'p.{seq3(value[0])}{value[1:-1]}{seq3(value[-1])}'

def prepare_alphamissense(input_filepath, output_filepath):
    am_class_transitions = {'pathogenic': 'P', 'ambiguous': 'A', 'benign': 'B'}

    with gzip.open(input_filepath, 'rt') as input_file, gzip.open(output_filepath, 'wt') as output_file:
        for row in input_file:
            row = row.rstrip('\r\n')
            if row and not row.startswith('#'):
                splitted_row = row.split('\t')
                if row.startswith('uniprot_id'):
                    splitted_row.insert(1, 'protein_position')
                    output_file.write('\t'.join(splitted_row) + '\n')
                else:
                    uniprot_id, protein_variant, am_pathogenicity, am_class = splitted_row
                    protein_position = extract_protein_position(protein_variant)
                    prepared_values = [uniprot_id, protein_position, to_hgvsp(protein_variant), round(float(am_pathogenicity), 2), am_class_transitions[am_class]]
                    output_file.write('\t'.join(str(value) for value in prepared_values) + '\n')


In [ ]:
def to_hgvsp(value):
    return f'p.{seq3(value[0])}{value[1:-1]}{seq3(value[-1])}'

#alphamissense_df = pd.read_csv(alphamissense_filepath, sep='\t', comment='#', usecols=['uniprot_id', 'protein_variant', 'am_pathogenicity'], dtype={'uniprot_id': 'category', 'protein_variant': 'string[pyarrow]', 'am_pathogenicity': 'float32'})
alphamissense_df = pd.read_csv(alphamissense_filepath, sep='\t', comment='#', usecols=['uniprot_id', 'protein_variant', 'am_pathogenicity', 'am_class'], dtype={'uniprot_id': 'category', 'am_pathogenicity': 'float32', 'am_class': 'category'})

In [ ]:
alphamissense_df

In [ ]:
alphamissense_df['am_class'].value_counts()

In [ ]:
alphamissense_df.memory_usage(deep=True).sum() / 1024**2